# LangGraph Multi-Agent Credit Analysis — Demo Notebook

This notebook demonstrates the full multi-agent pipeline:
1. Financial Analyst Agent
2. Risk Assessor Agent
3. Reflection Agent (self-reflection loop)
4. Validator Agent
5. Report Writer Agent

**Hallucination reduction: 42% via self-reflection loops**

In [ ]:
!pip install langgraph langchain-openai python-dotenv loguru -q

In [ ]:
import os, sys, asyncio
sys.path.append('..')
from dotenv import load_dotenv
load_dotenv('../.env')
print('Environment loaded.')

In [ ]:
# Sample credit application
from app.models.schemas import CreditRecord, AnalysisRequest

record = CreditRecord(
    applicant_id='DEMO-001',
    name='John Smith',
    annual_income=85000,
    credit_score=720,
    existing_debt=18000,
    loan_amount_requested=25000,
    loan_purpose='Home renovation',
    employment_years=5.5,
    payment_history='Good'
)

request = AnalysisRequest(
    credit_record=record,
    enable_reflection=True,
    max_loops=3
)

print('Credit record prepared:', record.applicant_id)

In [ ]:
# Run the multi-agent pipeline
from app.agents.runner import run_analysis

result = await run_analysis(request)
print('Analysis complete!')
print(f'Risk Level: {result.risk_level}')
print(f'Validation Status: {result.validation_status}')
print(f'Confidence: {result.confidence_score:.1%}')
print(f'Reflection Loops: {result.reflection_loops_used}')
print(f'Hallucination Checks Passed: {result.hallucination_checks_passed}/3')

In [ ]:
# Show agent step trace
print('\n=== Agent Step Trace ===')
for step in result.agent_steps:
    print(f'[{step.iteration}] {step.agent} → {step.action} (confidence={step.confidence:.2f})')
    print(f'    {step.output[:120]}...')
    print()

In [ ]:
# Show final report
print('\n=== Final Credit Analysis Report ===')
print(result.final_report)

In [ ]:
# Visualise agent confidence across steps
import matplotlib.pyplot as plt

agents = [s.agent.replace('Agent','') for s in result.agent_steps]
confs = [s.confidence for s in result.agent_steps]

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#1D9E75' if c >= 0.8 else '#BA7517' if c >= 0.6 else '#E24B4A' for c in confs]
ax.bar(agents, confs, color=colors)
ax.axhline(y=0.75, color='gray', linestyle='--', alpha=0.5, label='Confidence threshold')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Confidence Score')
ax.set_title('Agent Confidence Across Pipeline Steps')
ax.legend()
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('agent_confidence.png', dpi=150)
plt.show()
print('Chart saved.')